# Form persistence — does a player's form carry over week to week?
_Read-only diagnostic: within-player lag-1 rank autocorrelation of each signal and of returns, per position, plus the **observed** return rate the gameweek after a haul versus the base rate. **Describes serial structure in the observed season — not a forecast.** DGW excluded._

**Sections:** (a) within-player lag-1 autocorrelation · (b) after a haul: observed next-GW return rate vs base

---

## Setup
> Whole season, `minutes > 0`, **DGW excluded**; per position. Lag-1 pairing is by gameweek number (a missed GW yields no pair) via `within_player_autocorr`, which **demeans each value by the player's own season mean** so the autocorrelation reflects within-player form, not between-player class. The haul -> return transition via `post_event_outcome_rate`. A haul is `total_points > 12` (domain `HAUL_THRESHOLD_PTS`); a return is `total_points >= 4`. Both reads describe serial dependence already present in the season — neither is a prediction.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from dal.pipeline import load as load_mart, run as run_pipeline
from dal.exceptions import MartNotBuiltError, MartSchemaError
from research.kernels.descriptive.relevance import (
    compute_relevance, leading_indicator_signals, leading_alive_signals, POSITIONS,
)
from research.kernels.diagnostic.serial import within_player_autocorr, post_event_outcome_rate
from domain.registry.association import HAUL_THRESHOLD_PTS

RETURN_PTS = 4  # FPL convention: a 4+ point gameweek is a 'return'

POSITIONS = list(POSITIONS)
COLOURS = {"GK": "#9467bd", "DEF": "#1f77b4", "MID": "#2ca02c", "FWD": "#d62728"}

try:
    _r = load_mart()
except (MartNotBuiltError, MartSchemaError) as _e:
    print(f"Rebuilding mart ({type(_e).__name__})...")
    run_pipeline(force=True)
    _r = load_mart()

mart = _r.mart
df = mart[mart["gw"].between(1, _r.data_cutoff_gw)].copy()
df = df[df["minutes"].notna() & (df["minutes"] > 0)].copy()
df = df[df["is_dgw"] == False].copy()

leading = sorted(leading_indicator_signals(drop_exact_composites=True))
alive_by_pos = {}
for p in POSITIONS:
    rel = compute_relevance(df[df["position"] == p], signals=leading, group_cols=())
    alive_by_pos[p] = leading_alive_signals(rel)

print(f"Study range: GW 1 - {_r.data_cutoff_gw} | minutes > 0 | DGW excluded | n = {len(df):,}")
for p in POSITIONS:
    print(f"  {p}: n={len(df[df.position == p]):>6,} | {len(alive_by_pos[p])} live signals")

## (a) Within-player lag-1 autocorrelation
> For the same player — *relative to their own season average* — how strongly does this gameweek's value relate to last gameweek's? Positive = above-average weeks tend to follow above-average weeks (form is 'sticky'); near zero = each week is roughly independent of the last; negative = it tends to reverse. Demeaning strips out between-player class, so this is form, not quality (it mirrors the within component of `quality_vs_form`).

In [ ]:
ac_rows = []
for p in POSITIONS:
    d = df[df["position"] == p]
    ac_rows.append({"position": p, "signal": "total_points", **within_player_autocorr(d, "total_points")})
    for sig in alive_by_pos[p]:
        ac_rows.append({"position": p, "signal": sig, **within_player_autocorr(d, sig)})
ac = pd.DataFrame(ac_rows)
display(ac.sort_values(["position", "rho"], ascending=[True, False]).round(4))

fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharex=True)
for ax, p in zip(axes, POSITIONS):
    sub = ac[(ac.position == p) & ac["support_flag"].eq("")].sort_values("rho")
    colours = ["black" if s == "total_points" else COLOURS[p] for s in sub["signal"]]
    ax.barh(sub["signal"], sub["rho"], color=colours)
    ax.axvline(0, color="black", linewidth=0.5)
    ax.set_title(p)
    ax.set_xlabel("lag-1 within-player rho")
    ax.tick_params(axis="y", labelsize=7)
fig.suptitle("Does a player's own value persist week to week? (black = total_points; positive = sticky)", y=1.03)
plt.tight_layout()
plt.show()

## (b) After a haul: observed next-GW return rate vs base
> Among gameweeks where a player hauled (`total_points > 12`), how often did they return (`>= 4`) the next gameweek — versus the unconditional base rate? `lift` is the gap. Hauls are rare, so a position with fewer than 30 haul events is suppressed (`support_flag`) rather than reported on thin data. This is an observed coincidence rate in the season, not a forecast.

In [ ]:
dh = df.assign(is_haul=df["total_points"] > HAUL_THRESHOLD_PTS, is_return=df["total_points"] >= RETURN_PTS)
tr_rows = []
for p in POSITIONS:
    tr_rows.append({"position": p, **post_event_outcome_rate(dh[dh["position"] == p], "is_haul", "is_return")})
tr = pd.DataFrame(tr_rows)
display(tr.round(4))

ok = tr[tr["support_flag"].eq("")]
if len(ok):
    fig, ax = plt.subplots(figsize=(8, 3.4))
    x = np.arange(len(ok))
    w = 0.38
    ax.bar(x - w / 2, ok["base_rate"], w, color="#bdbdbd", label="base return rate")
    ax.bar(x + w / 2, ok["conditional_rate"], w, color=[COLOURS[p] for p in ok["position"]], label="after a haul")
    for i, (_, r) in enumerate(ok.iterrows()):
        ax.text(i + w / 2, r["conditional_rate"], f"n={int(r['n_event'])}", ha="center", va="bottom", fontsize=8)
    ax.set_xticks(x)
    ax.set_xticklabels(ok["position"])
    ax.set_ylabel("P(return next GW)")
    ax.set_ylim(0, 1)
    ax.set_title("Return rate the GW after a haul vs base rate")
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()
else:
    print("All positions below the haul-event floor; nothing to plot (see support_flag).")